In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib scikit-learn tqdm
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q librosa soundfile tensorboard
!apt-get install -qq ffmpeg sox
print('✅ 环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU不可用，请在运行时类型中选择T4 GPU')

# MLP与反向传播

## 学习目标

- 理解为什么需要激活函数（非线性）
- 理解多层感知机(MLP)的结构
- 理解反向传播的直觉
- 能用PyTorch搭建MLP并训练
- 完成第一个真正的训练任务：正弦波分类


## 1. 为什么需要激活函数？

如果没有激活函数，无论多少层线性变换叠加，结果仍然是线性的——跟一层没有区别。

**物理直觉**：人对声音响度的感知不是线性的，而是对数的。这就是一种非线性变换！
CI的频率-电极映射也不是线性的——低频区域分配更多电极。这又是一种非线性。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 4)

# 常见激活函数
x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 4, figsize=(16, 3))

# ReLU
axes[0].plot(x, np.maximum(0, x))
axes[0].set_title('ReLU')
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axvline(x=0, color='k', linewidth=0.5)
axes[0].grid(True, alpha=0.3)

# Sigmoid
axes[1].plot(x, 1 / (1 + np.exp(-x)))
axes[1].set_title('Sigmoid')
axes[1].grid(True, alpha=0.3)

# Tanh
axes[2].plot(x, np.tanh(x))
axes[2].set_title('Tanh')
axes[2].grid(True, alpha=0.3)

# LeakyReLU
axes[3].plot(x, np.where(x > 0, x, 0.01 * x))
axes[3].set_title('LeakyReLU')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### ReLU ≈ 半波整流

对学过信号处理的人来说，ReLU其实就是**半波整流器**！

$$\text{ReLU}(x) = \max(0, x)$$

信号处理中的半波整流：把负半周全部截掉，只保留正半周。ReLU做的完全一样的事。
这就是深度学习和信号处理的连接——很多概念你其实已经知道了，只是名字不同。

## 2. MLP的结构

多层感知机(MLP) = 多层线性变换 + 激活函数

```
输入 → 线性层 → 激活 → 线性层 → 激活 → ... → 输出
```

关键点：
- 每一层都是独立的线性变换+非线性激活
- 层数越多，模型能表达的函数越复杂（表达能力越强）
- 但层太多也可能过拟合

In [ ]:
# 用PyTorch定义MLP
import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # 输入层 → 隐藏层
            nn.ReLU(),                          # 激活函数
            nn.Linear(hidden_dim, hidden_dim),   # 隐藏层 → 隐藏层
            nn.ReLU(),                          # 激活函数
            nn.Linear(hidden_dim, output_dim),   # 隐藏层 → 输出层
        )
    
    def forward(self, x):
        return self.layers(x)

# 创建模型
model = SimpleMLP(input_dim=160, hidden_dim=64, output_dim=2)
print(model)
print(f'\n参数总数: {sum(p.numel() for p in model.parameters()):,}')

## 3. 反向传播的直觉

反向传播(backpropagation)就是**自动计算梯度**的具体算法。

直觉理解：
1. 前向传播：数据从输入层流向输出层，得到预测值
2. 计算损失：预测值和真实值的差距
3. 反向传播："误差信号"从输出层反向流回输入层
4. 每个参数问自己："我该变大还是变小才能减少误差？"

你不需要知道链式法则的数学细节——PyTorch的 `autograd` 帮你做了。
你只需要理解：**`loss.backward()` 会计算所有参数的梯度，`optimizer.step()` 会用这些梯度更新参数。**

In [ ]:
# 反向传播可视化
model = SimpleMLP(input_dim=160, hidden_dim=64, output_dim=2)
x = torch.randn(4, 160)  # batch_size=4, input_dim=160
y = torch.randint(0, 2, (4,))  # 4个标签

# 前向传播
output = model(x)
loss = nn.CrossEntropyLoss()(output, y)
print(f'损失: {loss.item():.4f}')

# 查看梯度（反向传播前）
print(f'\n反向传播前，第一层权重梯度: {model.layers[0].weight.grad}')

# 反向传播
loss.backward()

# 查看梯度（反向传播后）
print(f'反向传播后，第一层权重梯度: {model.layers[0].weight.grad}')
print(f'梯度形状: {model.layers[0].weight.grad.shape}')
print(f'权重形状: {model.layers[0].weight.shape}')

## 4. 第一个训练任务：正弦波分类

现在我们来做一个真正的训练任务：**分类两个不同频率的正弦波**。

- 输入：一段波形（160个采样点）
- 输出：低频(0) 或 高频(1)

这是最简单的分类任务——数据可以代码生成，模型2-3层MLP就够了，训练10秒就能收敛。

In [ ]:
# 生成数据
def generate_sine_wave(freq, duration=0.01, sr=16000, amplitude=0.5):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    return amplitude * np.sin(2 * np.pi * freq * t)

# 低频和高频正弦波
low_freq = 200   # 低频类别
high_freq = 800  # 高频类别
num_samples = 160  # 每个样本的采样点数

# 生成训练数据
n_train = 500
X_train = []
y_train = []
for _ in range(n_train // 2):
    # 低频 + 少量随机扰动
    f_low = low_freq + np.random.uniform(-50, 50)
    X_train.append(generate_sine_wave(f_low)[:num_samples])
    y_train.append(0)
    # 高频 + 少量随机扰动
    f_high = high_freq + np.random.uniform(-50, 50)
    X_train.append(generate_sine_wave(f_high)[:num_samples])
    y_train.append(1)

X_train = torch.FloatTensor(np.array(X_train))
y_train = torch.LongTensor(y_train)

print(f'训练数据形状: {X_train.shape}')  # [500, 160]
print(f'标签形状: {y_train.shape}')      # [500]
print(f'类别0(低频)样本数: {(y_train == 0).sum()}')
print(f'类别1(高频)样本数: {(y_train == 1).sum()}')

In [ ]:
# 可视化一些样本
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
t_axis = np.arange(num_samples) / 16000

# 低频
idx_low = (y_train == 0).nonzero(as_tuple=True)[0][0]
axes[0].plot(t_axis * 1000, X_train[idx_low].numpy())
axes[0].set_title('低频正弦波 (类别0)')
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# 高频
idx_high = (y_train == 1).nonzero(as_tuple=True)[0][0]
axes[1].plot(t_axis * 1000, X_train[idx_high].numpy())
axes[1].set_title('高频正弦波 (类别1)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 训练MLP分类器

现在用MLP来训练分类器。你需要填空完成训练循环。

**填空任务**：完成训练循环中的关键步骤。

In [ ]:
# ====== 练习：训练MLP分类器 ======

model = SimpleMLP(input_dim=160, hidden_dim=64, output_dim=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 训练
losses = []
accuracies = []
n_epochs = 50

for epoch in range(n_epochs):
    # TODO: 前向传播
    outputs = model(X_train)
    
    # TODO: 计算损失
    loss = criterion(outputs, y_train)
    
    # TODO: 反向传播（三步：清零梯度 → 反向传播 → 更新参数）
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 记录
    losses.append(loss.item())
    with torch.no_grad():
        predictions = outputs.argmax(dim=1)
        accuracy = (predictions == y_train).float().mean()
        accuracies.append(accuracy.item())
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}, Loss: {loss.item():.4f}, Acc: {accuracy.item():.4f}')

print(f'\n最终准确率: {accuracies[-1]:.4f}')

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(accuracies)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 挑战任务

1. 修改 `hidden_dim`，观察对训练速度和准确率的影响（试试16, 64, 256）
2. 修改学习率，观察训练曲线的变化（试试0.0001, 0.001, 0.01）
3. 增加一个隐藏层，观察是否提升性能
4. 尝试将低频和高频的频率靠近（如400 vs 600），观察分类是否变难

这些实验只需要改1-2行代码，但能让你对超参数的影响建立直觉。

## 本节要点

1. **激活函数**引入非线性——没有它，多层线性变换等于一层
2. **ReLU = 半波整流**——你已经懂的信号处理概念！
3. **MLP = 多层线性变换 + 激活函数**，是深度网络的基本构建块
4. **反向传播** = 自动计算梯度，你只需要 `loss.backward()` + `optimizer.step()`
5. **训练循环**：前向传播 → 计算损失 → 清零梯度 → 反向传播 → 更新参数


---
下一节：[03-cnn-audio.ipynb](03-cnn-audio.ipynb) — CNN与声音的"图像化"
